# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — base + GPU extras + browser agent
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
# Kaggle's base image already includes a CUDA torch; install the GPU
# extras NOT pinned to a specific torch wheel (skip torch line to
# avoid clobbering Kaggle's preinstalled CUDA torch).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
# Playwright + beautifulsoup for the NIM-driven research agent.
# `playwright install chromium` downloads the browser binary (~150 MB,
# usually <30 sec on Kaggle's fast network). Kaggle's runner has no
# system Chrome so we let Playwright manage its own.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-browser.txt'])
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('playwright + chromium ready — NIM research agent will activate on fact_research channels')

In [ ]:
# 3) BOOTSTRAP secrets.
#
# Two sources, in priority order:
#
#   (1) A private Kaggle Dataset attached via kernel-metadata.json. This
#       is the production path — survives `kaggle kernels push` cycles
#       (each push creates a new kernel version, but the dataset stays
#       attached). Create one private Dataset named e.g.
#       "yt-agent-secrets" with ONE file:
#
#           secrets.env
#               COOLIFY_BASE_URL=https://yt-agent.thyker.online
#               PB_URL=https://yt-agent.thyker.online/pb
#               POCKETBASE_ADMIN_EMAIL=admin@yt-agent.thyker.online
#               POCKETBASE_ADMIN_PASSWORD=your-password
#               RENDER_TRIGGER_KEY=...
#               STORAGE_PROVIDERS_ENC_KEY=...
#
#       Then reference it in kernel-metadata.json:
#           "dataset_sources": ["<your-username>/yt-agent-secrets"]
#
#   (2) Kaggle's Secrets panel (Add-ons → Secrets). Useful for
#       interactive testing. Same key names as the .env file.
#
# Either source works — the notebook reads whichever it finds first.
import os, glob

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    # Legacy Vercel + Firestore path — keep working for users who
    # haven\'t migrated to Coolify yet.
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]
ALL_KEYS = REQUIRED + OPTIONAL


def _load_from_dataset_envfile():
    """Look for an attached Dataset secrets file. Kaggle mounts datasets
    under /kaggle/input/<dataset-name>/. Search for the most common
    file names."""
    candidates = []
    for root in glob.glob('/kaggle/input/*/'):
        for name in ('secrets.env', '.env', 'env.txt', 'secrets.txt'):
            p = os.path.join(root, name)
            if os.path.exists(p):
                candidates.append(p)
    if not candidates:
        return False
    found = False
    for path in candidates:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ALL_KEYS and v:
                    os.environ.setdefault(k, v)
                    found = True
        print(f'  loaded secrets from {path}')
    return found


def _load_from_kaggle_secrets():
    """Read from the Kaggle Secrets panel. Fails silently when not
    available (e.g. running in a Colab clone of the notebook)."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        loaded = 0
        for k in ALL_KEYS:
            try:
                v = secrets.get_secret(k)
            except Exception:
                continue
            if v and not os.environ.get(k):
                os.environ[k] = v
                loaded += 1
        if loaded:
            print(f'  loaded {loaded} secrets from Kaggle Secrets panel')
        return loaded > 0
    except Exception as e:
        print(f'  Kaggle Secrets unavailable ({e!r}); using Dataset only')
        return False


# Try both sources — Dataset first, then Kaggle Secrets fills any gaps.
got_dataset = _load_from_dataset_envfile()
got_secrets = _load_from_kaggle_secrets()
if not (got_dataset or got_secrets):
    raise SystemExit(
        'No secrets source found. Either attach the yt-agent-secrets '
        'Dataset (via kernel-metadata.json dataset_sources) OR add the '
        'required keys to the Kaggle Secrets panel.'
    )

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))

# Defaults for the new outbound-poll mode.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Idle auto-shutdown ~25 min — preserves the 30 GPU-hr/week budget.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

print('Kaggle bootstrap OK')
print('  Dashboard:', os.environ.get('COOLIFY_BASE_URL'))
print('  Pocketbase:', os.environ.get('PB_URL'))
print('  Mode:', os.environ.get('WORKER_MODE'), '| tier:', os.environ.get('INSTANCE_TIER'))


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])